In [ ]:
#Achaibou Samy et Mouna Guehairia LDDBI (groupe 4)

In [ ]:
#bibliothèques

import json
import re
from textblob import TextBlob
import gradio as gr 
import pandas as pd 
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import torch
import numpy as np
from scipy.special import expit

In [ ]:

#Fichiers

fichier_tweet = "/Users/mouna.g/Desktop/projet-IN304/versailles_tweets_100.json"

#Fonctions

def nettoyage(fichier):
    with open(fichier, 'r',encoding='utf-8') as f:
        lignes = f.readlines()

    lignes_sans_caracteres = []
    for ligne in lignes:
        l = re.sub(r'[^\w\s]', '', ligne)
        lignes_sans_caracteres.append(l)

 
    chemin_fichier_nettoye = '/Users/mouna.g/Desktop/projet-IN304/zone_d_atterrissage.txt'
    with open(chemin_fichier_nettoye, "w") as fichier_nettoye:
        fichier_nettoye.writelines(lignes_sans_caracteres)

    return chemin_fichier_nettoye


def extraction_username(fichier):
    liste_utilisateurs=[]
    with open(fichier,'r', encoding='latin-1') as filin:
            tweets=json.load(filin)
            for tweet in tweets:
                if tweet.get('author_id'):
                    liste_utilisateurs.append(tweet["author_id"])
    return liste_utilisateurs



def extraction_hashtags(fichier):
    """Parcours les tweets et recupère les noms des differents hashtags trouvés"""
    liste_hashtags=[]
    with open(fichier,'r',encoding='utf-8') as filin:
       tweets=json.load(filin)
       for tweet in tweets:
           if 'text' in tweet:
               hashtags=re.findall(r"#\w+", tweet['text'])
               liste_hashtags.extend(hashtags)

    return liste_hashtags


def analyse_sentiment(fichier):
    liste_tweets_positifs=[]

    with open(fichier,'r', encoding='utf-8') as filin:
        tweets=json.load(filin)
        for tweet in tweets:
            if 'text' in tweet:
                texte=tweet['text']
                blob=TextBlob(texte)
                polarite=blob.sentiment.polarity
                if polarite > 0.0:
                    liste_tweets_positifs.append(texte)
        return liste_tweets_positifs


def extraction_name_description(fichier) :
    '''Fonction qui extrait les noms d'utilistauers et la bio associé a leurs comptes'''
    liste_name = []
    liste_description = []
    dico_name_description = {}
    with open(fichier, 'r', encoding= 'latin-1' ) as filin :
        for ligne in filin : 
            recherche_name = re.search(r'name\s+(\w+)', ligne)
            if recherche_name :
                liste_name.append(recherche_name.group(1))
            recherche_description = re.search(r'description\s+(.+)', ligne)
            if recherche_description :
                liste_description.append(recherche_description.group(1))
        dico_name_description = dict(zip(liste_name, liste_description))
        

    print(dico_name_description)
    return dico_name_description


def k_hashtags(tweets,k):
    '''Fonction va affichier les top hashtags'''
    hashtags=[]
    for tweet in tweets:
        if "text" in tweet:
            texte=tweet['text']
            hashtags+=re.findall(r"#\w+", texte)
    compteur_hashtags=Counter(hashtags)
    top_k_hashtag=compteur_hashtags.most_common(k)
    print("Les tops", k, "hashtags sont", top_k_hashtag)
    return top_k_hashtag

def k_utilisateurs(fichier,k):
    '''Fonction va affichier les utilisateurs les plus présents'''
    utilisateurs=extraction_username(fichier)
    compteur_utilisateurs=Counter(utilisateurs)
    top_k_utilisateurs=compteur_utilisateurs.most_common(k)
    print("Les tops", k, "utilisateurs sont", top_k_utilisateurs)
    return top_k_utilisateurs

def k_utilisateurs_mentionnes(tweets,k):
    '''Fonction va afficher les utilisateurs les plus mentionnés'''
    utilisateurs_mentionnes=[]
    for tweet in tweets:
        if "text" in tweet:
            texte=tweet['text']
            utilisateurs_mentionnes+=re.findall(r"@\w+", texte)
    compteur_utilisateurs_mentionnes=Counter(utilisateurs_mentionnes)
    top_k_utilisateurs_mentionnes=compteur_utilisateurs_mentionnes.most_common(k)
    print("Les tops", k, "utilisateurs mentionnes sont", top_k_utilisateurs_mentionnes)
    return top_k_utilisateurs_mentionnes



def k_topics(fichier,k):
    sujets=topics(fichier)
    compteur_topics=Counter(sujets)
    top_k_topics=compteur_topics.most_common(k)
    print("Les tops", k, "topics", top_k_topics)
    return top_k_topics




def topics(fichier):
    """Code inspiré par la documentation trouvée sur https://huggingface.co/cardiffnlp/tweet-topic-21-multi, 
    Affiche les topics des différents tweets"""

    model_name = "cardiffnlp/tweet-topic-21-multi"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    # Mapping des classes
    class_mapping = model.config.id2label
    liste_topics=[]


    with open('/Users/mouna.g/Desktop/projet-IN304/versailles_tweets_100.json', 'r', encoding='utf-8') as fichier:
     sujets = json.load(fichier)

    for tweet in sujets:
     if "text" in tweet:
        texte = tweet["text"]
        print("Le tweet est", {texte})

        inputs = tokenizer(texte, return_tensors="pt", padding=True, truncation=True)

        with torch.no_grad():
            output = model(**inputs)
            scores = output.logits[0].detach().numpy()
            probabilities = expit(scores)  

        predictions = (probabilities >= 0.5).astype(int)


        for i, prediction in enumerate(predictions):
            if prediction == 1:
             print("Le sujet predit est",class_mapping[i])
             liste_topics.append(class_mapping[i])

    return liste_topics

    
              
def nombre_publications_utilisateur(fichier):
    """Affiche le nombre de tweets par uitlisateur"""
    compteur=Counter()
    with open(fichier,'r',encoding='utf-8') as filin:
        tweets=json.load(filin)
        for tweet in tweets:
            author_id=tweet.get('author_id')
            if author_id:
               compteur[author_id]+=1
    return compteur  
    
def nbr_publications_hashtags(fichier):
    """Affiche le nombre de publications par hashtag"""
    liste_des_hashtags=extraction_hashtags(fichier)
    compteur_hashtags=Counter()
    for hashtags in liste_des_hashtags:
             compteur_hashtags[hashtags]+=1
    
    return compteur_hashtags


def nombre_publications_topics(fichier):
     """Affiche le nombre de publications par topic"""
     liste_des_topics=topics(fichier)
     compteur_topics=Counter(liste_des_topics)
  

     return compteur_topics
     
def tweets_utilisateur_specifique(fichier,utilisateur):
   """Affiche tous les tweets écrits par un utilisateur donné en paramètre """
   liste_tweets_utilisateurs_donnes=[]
   with open(fichier,'r',encoding='utf-8') as filin:
       tweets=json.load(filin)
       for tweet in tweets:
           if tweet.get('author_id') ==utilisateur:
              liste_tweets_utilisateurs_donnes.append(tweet["text"])
   return liste_tweets_utilisateurs_donnes



def tweets_utilisateur_identifie(fichier,utilisateur_mentionne):
    """Affiche tous les tweets qui mentionnent un utilisateur donné en paramètre"""
    liste_tweets_utilisateurs_identifies=[]

    with open(fichier,'r',encoding='utf-8') as filin:
       tweets=json.load(filin)
       for tweet in tweets:
           if "entities" in tweet:
              if "mentions" in tweet["entities"]:
                 for mention in tweet["entities"]["mentions"]:
                  if mention["id"] ==utilisateur_mentionne:
                     liste_tweets_utilisateurs_identifies.append(tweet["text"])
    return liste_tweets_utilisateurs_identifies



def utilisateurs_mentionnant_hashtag(fichier, hashtag_choisi):
   """Affiche tous les utilisateurs mentionnant un hashtag choisi en paramètre"""
   liste_utilisateurs_mention_hashtag=[]
   with open(fichier,'r',encoding='utf-8') as filin:
       tweets=json.load(filin)
       for tweet in tweets:
          if 'text' in tweet:
           hashtags=re.findall(r"#\w+", tweet['text'])
           if hashtag_choisi in hashtags:
                liste_utilisateurs_mention_hashtag.append(tweet["author_id"])
   return liste_utilisateurs_mention_hashtag
                

def utilisateurs_mentionnes(fichier,utilisateur):
   """Affiche les utilisateurs mentionnés par un utilisateur"""
   liste_utilisateurs_identifies=[]
   with open(fichier,'r',encoding='utf-8') as filin:
       tweets=json.load(filin)
       for tweet in tweets:
           if tweet.get('author_id') ==utilisateur:
              if "entities" in tweet:
               if "mentions" in tweet["entities"]:
                 for mention in tweet["entities"]["mentions"]:
                   liste_utilisateurs_identifies.append(mention["id"])
   return liste_utilisateurs_identifies


#Interface avec Gradio

def tableau(fichier) :
    dico_name_description = extraction_name_description(fichier)
    liste_pour_tableau = [(name, description) for name, description in dico_name_description.items()]
    interface1 = gr.Interface(
        fn=lambda: liste_pour_tableau,
        inputs=[],
        outputs=gr.DataFrame(headers=["Name", "Description"]),
        title= "Name and descirption extracted"
    )
    interface1.launch()

interface1=gr.Interface(fn=nettoyage, inputs=gr.File(label="Choisir un fichier"), outputs=gr.File(label="Voici le fichier nettoyé:"), title="Extraire les identifiants")

def option_chosie(fichier,choix, k=None, utilisateur=None, choix_hashtag=None):
    with open(fichier, 'r', encoding='utf-8') as filin:
        tweets=json.load(filin)

    if choix=="nettoyage":
        return nettoyage(fichier)
    if choix=="extraire les hashtags":
        return extraction_hashtags(fichier)
    elif choix=="extraire les utilisateurs":
        return extraction_username(fichier)
    elif choix=="sentiment de la publication":
        return analyse_sentiment(fichier)
    elif choix=="identifier les topics":
        return topics(fichier)
    elif choix=="top K hashtags":
        return k_hashtags(tweets,k)
    elif choix=="top K utilisateurs":
        return k_utilisateurs(fichier,k)
    elif choix=="top K utilisateurs mentionnés":
        return k_utilisateurs_mentionnes(tweets,k)
    elif choix=="top K topics":
        return k_topics(fichier,k)
    elif choix=="nombre de publications par utilisateur":
        return nombre_publications_utilisateur(fichier)
    elif choix=="nombre de publications par hashtag":
        return nbr_publications_hashtags(fichier)
    elif choix=="nombre de publications par topic":
        return nombre_publications_topics(fichier)
    elif choix=="utilisateur mentionnant le hashtag":
        return utilisateurs_mentionnant_hashtag(fichier,choix_hashtag)
    elif choix=="utilisateurs mentionnés par un utilisateur spécifique":
        return utilisateurs_mentionnes(fichier,utilisateur)
    else:
        return "Votre choix n'est pas proposé"
    

with gr.Blocks() as interface:
    with gr.Row():
        choix=gr.Radio(choices=["nettoyage","extraire les hashtags","extraire les utilisateurs","sentiment de la publication",
                                "identifier les topics","top K hashtags", "top K utilisateurs", "top K utilisateurs mentionnés",
                                "top K topics", "nombre de publications par utilisateur", "nombre de publications par hashtag", 
                                "nombre de publications par topic", "utilisateur mentionnant le hashtag",  
                                "utilisateurs mentionnés par un utilisateur spécifique"], label="Quelle analyse voulez-vous effecetuer sur votre fichier?")
        k_valeur=gr.Number(label="Choisir une valeur k")
        utilisateur=gr.Textbox(label="Choisir un utilisateur")
        choix_hashtag=gr.Textbox(label="Choisir un hashtag")
        uploader=gr.File(label="Télécharger votre fichier")
        valider_bouton=gr.Button("Lancer l'analyse")

    
    sortie=gr.Textbox(label="Résultats")
    valider_bouton.click(fn=option_chosie, inputs=[uploader,choix,k_valeur, utilisateur, choix_hashtag],outputs=sortie)



interface.launch()




In [ ]:

#Fonctions ecrites precedemmet

def k_hashtags(tweets,k):
    '''Fonction va affichier les top hashtags'''
    hashtags=[]
    for tweet in tweets:
        if "text" in tweet:
            texte=tweet['text']
            hashtags+=re.findall(r"#\w+", texte)
    compteur_hashtags=Counter(hashtags)
    top_k_hashtag=compteur_hashtags.most_common(k)
    return top_k_hashtag


def k_utilisateurs(fichier,k):
    '''Fonction va affichier les utilisateurs les plus présents'''
    utilisateurs=extraction_username(fichier)
    compteur_utilisateurs=Counter(utilisateurs)
    top_k_utilisateurs=compteur_utilisateurs.most_common(k)

    return top_k_utilisateurs

def k_utilisateurs_mentionnes(tweets,k):
    '''Fonction va afficher les utilisateurs les plus mentionnés'''
    utilisateurs_mentionnes=[]
    for tweet in tweets:
        if "text" in tweet:
            texte=tweet['text']
            utilisateurs_mentionnes+=re.findall(r"@\w+", texte)
    compteur_utilisateurs_mentionnes=Counter(utilisateurs_mentionnes)
    top_k_utilisateurs_mentionnes=compteur_utilisateurs_mentionnes.most_common(k)
    return top_k_utilisateurs_mentionnes


def k_topics(fichier,k):
    sujets=topics(fichier)
    compteur_topics=Counter(sujets)
    top_k_topics=compteur_topics.most_common(k)
    return top_k_topics


def nombre_publications_utilisateur(fichier):
    """Affiche le nombre de tweets par uitlisateur"""
    compteur=Counter()
    with open(fichier,'r',encoding='utf-8') as filin:
        tweets=json.load(filin)
        for tweet in tweets:
            author_id=tweet.get('author_id')
            if author_id:
               compteur[author_id]+=1
    return compteur  

#Afficher les graphes avec Matplotlib

def graph_top_hashtags(top_k_hashtags):
     """Creer un graphique montrant les top hashtags pour une valeur de K choisie"""
     hashtags,frequences=zip(*top_k_hashtags)
     plt.figure(figsize=(10,6))
     plt.barh(hashtags,frequences,color='blue')
     plt.xlabel("Les frequences")
     plt.ylabel("hashtags")
     plt.title("Top K hashtags")
     plt.gca().invert_yaxis()
     plt.show()


def graph_top_utilisateurs(top_k_utilisateurs):
     """Creer un graphique montrant les top utilisateurs pour une valeur de K choisie"""
     utilisateurs,frequences=zip(*top_k_utilisateurs)
     plt.figure(figsize=(10,6))
     plt.barh(utilisateurs,frequences,color='green')
     plt.xlabel("Les frequences")
     plt.ylabel("utilisateurs")
     plt.title("Top K utilisateurs")
     plt.gca().invert_yaxis()
     plt.show()    


def graph_top_utilisateurs(top_k_utilisateurs_mentionnes):
     """Creer un graphique montrant les top utilisateurs mentionnes pour une valeur de K choisie"""
     utilisateurs_mentionnes,frequences=zip(*top_k_utilisateurs_mentionnes)
     plt.figure(figsize=(10,6))
     plt.barh(utilisateurs_mentionnes,frequences,color='red')
     plt.xlabel("Les frequences")
     plt.ylabel("utilisateurs mentionnés")
     plt.title("Top K utilisateurs mentionnés")
     plt.gca().invert_yaxis()
     plt.show()    


def graph_k_topics(top_k_topics):
     """Creer un graphique montrant les top K topics"""
     topics,frequences=zip(*top_k_topics)
     plt.figure(figsize=(10,6))
     plt.barh(topics,frequences,color='red')
     plt.xlabel("Les frequences")
     plt.ylabel("Topics")
     plt.title("Top K topics")
     plt.gca().invert_yaxis()
     plt.show()    


def graph_nbr_publications_utilisateur(nbr_public_utilisateur):
     """Creer un graphique montrant le nombre de publications par utilisateur"""
     nombre_publications,frequences=zip(*nbr_public_utilisateur)
     plt.figure(figsize=(10,6))
     plt.barh(nombre_publications,frequences,color='red')
     plt.xlabel("Les frequences")
     plt.ylabel("Le nombre de publications")
     plt.title("Nombre de publications par utilisateur")
     plt.gca().invert_yaxis()
     plt.show()


fichier="/Users/mouna.g/Desktop/projet-IN304/versailles_tweets_100.json"
with open(fichier, 'r', encoding='utf-8') as filin:
        tweets=json.load(filin)

#top_k_hashtags=k_hashtags(tweets,5)
#graph_top_hashtags(top_k_hashtags)

#top_k_utilisateurs=k_utilisateurs(fichier,5)
#graph_top_utilisateurs(top_k_utilisateurs)

#top_k_utilisateurs_mentionnes=k_utilisateurs_mentionnes(tweets,5)
#graph_top_utilisateurs(top_k_utilisateurs_mentionnes)

#top_k_topics=k_topics(fichier,5)
#graph_k_topics(top_k_topics)

#nbr_publications_utilisateur=nombre_publications_utilisateur(fichier)
#graph_nbr_publications_utilisateur(nbr_publications_utilisateur)